# Ejercicio: Extracción de datos de la API de TMDB (The Movie Database)

![imagen](https://www.themoviedb.org/assets/2/v4/logos/v2/blue_square_2-d537fb228cf3ded904ef09b136fe3fec72548ebc1fea3fbbd1ad9e36364db38b.svg)

Trabajaremos con **TMDB**, una de las bases de datos de cine más importantes del mundo. El objetivo es extraer información de películas para realizar un análisis de mercado inicial.

### 1. Registro y Documentación
Tendrás que [registrarte en TMDB](https://www.themoviedb.org/signup) para obtener tu **API Key (v3 auth)** y consultar la [documentación oficial](https://developer.themoviedb.org/reference/intro/getting-started).

### 2. Objetivo del Ejercicio
Queremos que consultes la API para que te devuelva la información de las películas que empiecen por la **inicial de tu nombre** (parámetro `query`). 

Debes guardar la información en un archivo `.csv` con la siguiente estructura de columnas:

| Columna | Descripción |
| :--- | :--- |
| **id** | ID interno de la película en TMDB |
| **title** | Título de la película |
| **release_date** | Fecha de estreno |
| **genres** | Nombres de los géneros (ej: "Acción, Comedia") |
| **vote_average** | Puntuación media de los usuarios |
| **overview** | Sinopsis o resumen de la trama |

### 3. El Reto: Mapeo de Géneros
A diferencia de otras APIs, el endpoint de búsqueda de películas devuelve los géneros como una lista de IDs numéricos (ej: `[28, 12]`). 

**Tu labor es:**
1. Consultar el endpoint de "Genre List" para obtener la relación entre IDs y nombres.
2. Sustituir los IDs en tu DataFrame final por los nombres reales de los géneros (separados por comas).

---

### Código de inicio
Aquí tienes el bloque para empezar a configurar tus llamadas:

```python
import requests
import pandas as pd

# Rellena estas variables
api_key = "TU_API_KEY_AQUÍ"
url_base = "[https://api.themoviedb.org/3](https://api.themoviedb.org/3)"
mi_inicial = "P" # Sustituye por tu inicial

In [ ]:
import requests
import pandas as pd

# Configuración
api_key = "930590f441225ca054ca5c0d0a53220e"
url_base = "https://api.themoviedb.org/3"
mi_inicial = "u" # Cambiar por la inicial del alumno
genres = '/genre/movie/list?'
search = '/search/movie?'


# 1. Obtener el diccionario de géneros (ID -> Nombre)
# Tip: Endpoint /genre/movie/list
def get_genres_map(api_key):
    url = url_base + genres + 'api_key=' + api_key
    response = requests.get(url)

    if response.status_code == 200:  # Código 200 indica una respuesta exitosa.
        data = response.json()  # Analizar la respuesta JSON.
        return data
    else:
        print("Error en la solicitud: ", response.status_code)


# 2. Buscar películas
# Tip: Endpoint /search/movie
def search_movies(api_key, query, page = 1):
    # Metemos paginado para recuperar las películas de cada página
    url = url_base + search + 'query=' + query + '&page=' + str(page) + '&api_key=' + api_key
    # print(url) # imprimimos para pruebas
    response = requests.get(url)

    if response.status_code == 200:  # Código 200 indica una respuesta exitosa.
        data = response.json()  # Analizar la respuesta JSON.
        return data
    else:
        print("Error en la solicitud: ", response.status_code)
        return
    

def busca_pelis(api_key, query):
    # Búsqueda inicial de películas
    data = search_movies(api_key, query)
    
    # Recuperamos total páginas para saber si hay más de una
    pages = data['total_pages']
    lista_pelis = []
    if pages > 1:
        # Bucle de páginas
        for page in range(pages):
            
            lista_pelis = guardar_pelis(data, lista_pelis)
            
    else:
        lista_pelis = guardar_pelis(data, lista_pelis)

            
    return lista_pelis



# | **id** | ID interno de la película en TMDB |
# | **title** | Título de la película |
# | **release_date** | Fecha de estreno |
# | **genres** | Nombres de los géneros (ej: "Acción, Comedia") |
# | **vote_average** | Puntuación media de los usuarios |
# | **overview** | Sinopsis o resumen de la trama |

# 3. Procesamiento y Limpieza

def guardar_pelis(movies, lista_pelis):
    # Bucle por película de la página
    for peli in movies['results']:
        pel = {}
        pel['id'] = peli['id']
        pel['title'] = peli['title']
        pel['release_date'] = peli['release_date']
        pel['genres'] = peli['genres']
        pel['vote_average'] = peli['vote_average']
        pel['overview'] = peli['overview']
        lista_pelis.append(pel)
    return lista_pelis
    


# 'results': [{'adult': False,
#    'backdrop_path': '/zbrSxpgeNscm06Wz39qa9YP5h9m.jpg',
#    'genre_ids': [16, 14],
#    'id': 39405,
#    'original_language': 'fr',
#    'original_title': 'U',
#    'overview': "Since her parents passed away, Princess Mona lives by herself in a castle with two hideous, ghastly characters, Goomi and His Lordship. One day her sobs draw the attention of a unicorn, called U, who wants to comfort and protect Mona for as long as she needs. U becomes Mona's companion, her confidant and her inseparable friend. While Mona grows into a beautiful princess, a group of charming, peaceful and fanciful Yeah-Yeah’s settles in the neighboring forest. They have no particular special powers, yet they soon bring about major changes in everyone's lives, especially in Kulka, the dreamy musician.",
#    'popularity': 1.5228,
#    'poster_path': '/fPyDf7cl4NrtjSn5jnmODDF70kS.jpg',
#    'release_date': '2006-10-11',
#    'title': 'U',
#    'video': False,
#    'vote_average': 6.6,
#    'vote_count': 100},

#   {'adult': False,
#    'backdrop_path': '/O50sdY6JEwpU7cQRaFtTO9eSAk.jpg',
#    'genre_ids': [28],
#    'id': 1499984,
#    'original_language': 'en',
#    'original_title': 'Gladiator Underground',
#    'overview': "Two brothers compete in the world's deadliest underground fighting tournament, where they must decide whether to play by the rules or to cast their honor aside and team up to bring down the dark syndicate running the tournament.",
#    'popularity': 9.3456,
#    'poster_path': '/qLN16AeIjS5P5Kk5B6pBi1Mh1mi.jpg',
#    'release_date': '2025-08-22',



# - Mapear los IDs de géneros a nombres reales.
# - Crear el DataFrame.
# - Exportar a CSV.

In [24]:
for i in range(5):
    print(i)

0
1
2
3
4


In [ ]:
id_genres = get_genres_map(api_key)
id_genres

In [ ]:
# Guardamos los id de géneros
generos = {}

for genre in id_genres['genres']:
    # nombre = genre['name']
    generos[genre['name']] = genre['id']

generos
    


In [ ]:
# print('https://api.themoviedb.org/3/search/movie?query=u&page=1&api_key=930590f441225ca054ca5c0d0a53220e')
query = mi_inicial
url = f'{url_base}{search}query={query}&api_key={api_key}'

pelis = search_movies(api_key, query)
pelis


In [ ]:
# Hay más páginas y no consigo aumentar el límite por página
# Recuperamos el número de páginas totales para saber cuántas búsquedas debemos hacer
paginas = pelis['total_pages']
paginas


In [ ]:
# Creamos bucle para recuperar todas las páginas
movies = []
pelicula = {
    'id' : 0,
    'title' : '',
    'release_date' : '',
    'genres' : [],
    'vote_average' : 0.0,
    'overview' : ''
}

